# Pricing American Options

Whereas a European option can be exercised only at a fixed date, an American option can be exercised any time up to its expiration, thus, the value of an American option is the value achieved by exercising optimally.
The problem is to solve the following optimization problem:
$ sup_{\tau \in \Tau} \mathbb{E}[U(\tau)]$, where $\tau$ representa a stopping time taking value in $[0,T]$ and $U$ represents the discounted payoff from exercise at time t.

We begin by pricing an American Put with interest rate $r$ and divident yield $q$ constant and positive s.t. r > q, depending on only one asset which follows the dynamics of black-Scholes model, using the Longstaff–Schwartz method, a standard approach in the literature.

Therefore, consider a version of black-Scholes model in which the underlying asset has dividend yield $q$, then the risk-neutral dynamics of the asset is given by 
$$
\frac{dS_t}{S_t} = ( r - q ) dt + \sigma dW_t
$$

More specifically in the present simulation we are considering a bermudan option where every discretized time point is a possible exercise date.

In [1]:
import numpy as np
import matplotlib.pyplot as plt


In [2]:
# Utility function 
	
# Simulation of the underlying asset price using Geometric Brownian Motion (GBM)
def simulate_gbm( mu, sigma, T, dt, b, seed):
	np.random.seed(seed)
	N = int(T / dt)  # Number of time steps
	Z = np.random.standard_normal((b // 2, N)) # Increments
	Z = np.vstack([Z, -Z])
	BM = np.hstack([np.zeros((b,1)), np.cumsum(Z * np.sqrt(dt), axis=1)]) 
	time = np.arange(0, N + 1) * dt
	drift = (mu - 0.5 * sigma**2) * time
	S = np.exp(drift + sigma * BM)
	return S

# The value of the option starts at t_1, because we are assuming that the option can be exercised at t_1, t_2, ..., t_N (and not at t_0)

In [10]:
# Parameters
K = 100 # Strike price
sigma = 0.3 # Volatility 
r = 0.05  # Risk-free rate
q = 0 # Dividend yield
mu = r - q  # Drift 
T = 0.2  # Maturity 
dt = 0.004  # Time step (daily 1/252)
N = int(T / dt)  # Number of steps 
npaths_train= 100000 
npaths_pricing = 1000000
max_deg = 4 # Degree of the polynomial basis for regression
# Paths simulation for training and pricing 
train_set = simulate_gbm(mu, sigma, T, dt, npaths_train, seed=31415)
test_set = simulate_gbm( mu, sigma, T, dt, npaths_pricing, seed = 31416) 
# Transformed input 
train_log = np.log(train_set)
std_train = np.std(train_log, axis=0)
std_train[0] = 1.0 # FIX 
X_train_transformed = (train_log - np.mean(train_log, axis=0)) / std_train

test_log = np.log(test_set)
std_test = np.std(test_log, axis=0)
std_test[0] = 1.0 # FIX 
X_test_transformed = (test_log - np.mean(test_log, axis=0)) / std_test

# Computing the intrinsic value at time t discounted to time 0 when necessary
discount_factors = np.exp(-r * np.arange(0, N + 1) * dt)

for S0 in np.arange(60,141):
    # Training phase : Backward induction compute the coefficients
    value = np.maximum(0, K - S0*train_set[:,-1]) * discount_factors[-1] # Intrinsic value discounted to time 0
    train_deg = np.zeros(N-1, dtype=int) # Actual degree used for regression at each time step
    coefficients = np.zeros((N-1, max_deg + 1)) # Storing the coefficient for later use in the pricing phase
    for t in range(N - 1, 0, -1):
        payoff_t = np.maximum(0, K - S0*train_set[:,t]) * discount_factors[t] # Intrinsic value at time t discounted to time 0
        itm = payoff_t > 0 # Indices of ITM paths at time t, discounted to time 0
        itm_paths = sum(itm) # Number of in the money paths at time t

        X_transformed = X_train_transformed[:, t] # Transformed paths at time t
        X_itm = X_transformed[itm] # Only the in-the-money paths are used for regression
        Y_itm = value[itm] 

        # Regression to estimate continuation value, hence to compute the coefficient 
        train_deg[t-1] = min(max_deg, int(itm_paths / 100) ) # Degree of the polynomial basis for regression
        if (train_deg[t-1] > 0) & (itm_paths > train_deg[t-1]):
            # Vandermonde matrix for polynomial regression of degree train_deg[t-1]
            A = np.vander(X_itm, N=train_deg[t-1] + 1, increasing=True)
            c, _, _, _ = np.linalg.lstsq(A, Y_itm, rcond=None) # Coefficients of the polynomial regression
            coefficients[t-1, :train_deg[t-1]+1] = c # Storing the coefficients for later use in the pricing phase
            A_full = np.vander(X_transformed, N=train_deg[t-1] + 1, increasing=True)
            continuation_value = A_full @ c # Continuation value at time t for the in-the-money paths
            
            #Polyfit
            #coefficients[t-1, -(train_deg[t-1]+1):] = np.polyfit(X_itm, Y_itm, deg=train_deg[t-1])
            #continuation_value = np.polyval(coefficients[t-1,-(train_deg[t-1]+1):], X_transformed)

            # Determine the option value at time t
            exercise = (payoff_t > continuation_value) & (payoff_t > 0 ) # Exercise condition at time t
            # The option is early-exercised if the payoff is positive and greater than the continuation value!
            value[exercise] = payoff_t[exercise] # If the option is exercised, the value is the payoff

            #value = np.where( payoff_t > continuation_value, payoff_t, value)   

    # Pricing phase
    #continuation_value_pricing = np.zeros((npaths_pricing, N-1 ) ) # Storing only for plotting
    exercise_payoff = np.zeros(npaths_pricing) # Store the exercise time for each path, initialized to N (never early-exercised)
    has_exercised = np.zeros(npaths_pricing, dtype=bool) # Tracking if the option has been exercised for each path

    # Computing the price
    for t in range(0, N - 1 ):
        transformed_paths = X_test_transformed[:, t + 1] # Transformed paths at time t
        intrinsic_value = np.maximum(0, K - S0*test_set[:, t + 1]) * discount_factors[t+1] # Intrinsic value at time t+1 discounted to time 0
        itm_pricing = intrinsic_value > 0 # Indices of ITM paths at time t+1, discounted to time 0
        itm_pricing_paths = sum(itm_pricing) # Number of ITM paths 
        if (train_deg[t] > 0) & (itm_pricing_paths > train_deg[t]):
            # Vandermonde matrix for polynomial regression of degree train_deg[t]
            A = np.vander(transformed_paths, N=train_deg[t] + 1, increasing=True)
            continuation_value_pricing = A @ coefficients[t, :train_deg[t]+1] # Continuation value at time t for the in-the-money paths

            # Polyfit
            #continuation_value_pricing = np.polyval(coefficients[t,-(train_deg[t] + 1):], transformed_paths)
            #continuation_value_pricing[:,t] = np.polyval(coefficients[t,-(train_deg[t] + 1):], transformed_paths)
            exercise_condition = (intrinsic_value > continuation_value_pricing) & itm_pricing & (has_exercised == False)
            exercise_payoff[exercise_condition] = intrinsic_value[exercise_condition]
            has_exercised[exercise_condition] = True 
    # If the option has not been exercised at any time step, the payoff is the one at maturity  
    payoff_T = np.maximum(0, K - S0 * test_set[:, -1]) * discount_factors[-1]
    exercise_payoff[~has_exercised] = payoff_T[~has_exercised]

            #exercise_time = np.where((intrinsic_value > continuation_value_pricing) & (has_exercised == False), t+1, exercise_time)
            #has_exercised = np.where((intrinsic_value > continuation_value_pricing) & (has_exercised == False), True, has_exercised)
    option_price = np.mean(exercise_payoff)
    #option_price = np.maximum(0, K - S0*test_set[np.arange(npaths_pricing), exercise_time.astype(int)]) * discount_factors[exercise_time.astype(int)]
    immediate_exercise= max(0, K - S0) # Equivalent to np.mean(actualized_payoff_pricing[:,0])
    option_price = max(option_price, immediate_exercise)
    print(f"Estimated option price for S0={S0}: {option_price:.5f}")

Estimated option price for S0=60: 40.00000
Estimated option price for S0=61: 39.00000
Estimated option price for S0=62: 38.00000
Estimated option price for S0=63: 37.00000
Estimated option price for S0=64: 36.00000
Estimated option price for S0=65: 35.00000
Estimated option price for S0=66: 34.00000
Estimated option price for S0=67: 33.00000
Estimated option price for S0=68: 32.00000
Estimated option price for S0=69: 31.00000
Estimated option price for S0=70: 30.00000
Estimated option price for S0=71: 29.00000
Estimated option price for S0=72: 28.00000
Estimated option price for S0=73: 27.00000
Estimated option price for S0=74: 26.00000
Estimated option price for S0=75: 25.00000
Estimated option price for S0=76: 24.00000
Estimated option price for S0=77: 23.00000
Estimated option price for S0=78: 22.00000
Estimated option price for S0=79: 21.00000
Estimated option price for S0=80: 20.00000
Estimated option price for S0=81: 19.00000
Estimated option price for S0=82: 18.01509
Estimated o